# 4.4 — Programmatic scenes and waypoint goals

Build a scene, choose an actor, and specify ordered XY targets and an arrival radius.
This **draft CPU prototype** reuses `ModelBuilder` and `MyoGymnasiumEnv`. It is not a registered task; a matching mjlab implementation and parity tests are required before promotion to a supported task family.

**Run:** install this PR's checkout with `pip install -e .`, plus `pip install jupyter ipykernel`. Run this notebook from the repository or `tutorials/` directory. No policy checkpoints, Hugging Face login or TERRA installation are needed. The included TERRA video is a prior recorded solution, not generated by these cells.

Scene authoring is arbitrary within MuJoCo's `MjSpec` capabilities. Physical solvability and controller compatibility must be checked separately.

In [ ]:
import os
import sys
from pathlib import Path

if sys.platform.startswith("linux") and not os.environ.get("DISPLAY"):
    os.environ.setdefault("MUJOCO_GL", "egl")

import gymnasium as gym
import mujoco
import myo_sim
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Video, display
from myosuite.core.model_builder import ModelBuilder
from myosuite.envs.waypoint import WaypointEnv

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "tutorials").is_dir())
assets = root / "tutorials" / "files" / "4.4"

## 1. Define a scene programmatically

Here stairs, beams and landing pads are ordinary static MuJoCo box geoms. `size` uses **half extents** and `pos` is the centre in metres. Explicit collision flags prevent inheriting non-colliding visual defaults from the actor. Use rotated boxes, meshes, heightfields or dynamic bodies through the same `MjSpec` API as needed; this example only validates static boxes.

The recipe below is independent of the actor and controller. Change its dimensions or replace it entirely.

In [ ]:
def course_scene(spec: mujoco.MjSpec, rise: float = .06,
                 beam_width: float = .36, gap: float = .20) -> mujoco.MjSpec:
    """Add static terrain to a scene containing a floor and actor."""
    if not np.isfinite([rise, beam_width, gap]).all() or min(rise, beam_width, gap) <= 0:
        raise ValueError("Course dimensions must be finite and positive")

    def box(name: str, x: float, y: float, width: float, length: float, height: float) -> None:
        spec.worldbody.add_geom(
            name=name, type=mujoco.mjtGeom.mjGEOM_BOX,
            pos=[x, y, height / 2], size=[width / 2, length / 2, height / 2],
            contype=1, conaffinity=1, condim=3, friction=[.8, .005, .0001],
            rgba=[.15, .45, .55, 1],
        )

    for i in range(4):
        box(f"course_up_{i}", 0, -.8 - (i + .5) * .3, 1, .3, (i + 1) * rise)
    height = 4 * rise
    box("course_top", 0, -2.3, .8, .6, height)
    box("course_beam_east", 1.1, -2.3, 1.4, beam_width, height)
    box("course_turn", 2.1, -2.3, .6, .8, height)
    box("course_beam_south", 2.1, -3, beam_width, .6, height)
    box("course_jump_landing", 2.1, -3.6 - gap, .8, .6, height)
    for i in range(3):
        box(f"course_down_{i}", 2.1, -4.05 - gap - i * .3, 1, .3, (3 - i) * rise)
    box("course_jump2_landing", 2.1, -5.1 - 2 * gap, .8, .6, rise)
    return spec

## 2. Choose the actor and compile

Use the packaged MyoFullBody actor. `ModelBuilder.from_spec` and `apply_transform` compose it with the scene before compilation. This native actor is **not checkpoint-compatible by assumption** with TERRA's saved actor configuration. Muscle conditions or another actor belong in model construction, not in the waypoint evaluator.

In [ ]:
actor_spec = myo_sim.load_spec("myofullbody")
actor_model = actor_spec.compile()
model, scene_spec = (
    ModelBuilder.from_spec(actor_spec)
    .apply_transform(course_scene)
    .build()
)
assert (model.nq, model.nv, model.nu) == (actor_model.nq, actor_model.nv, actor_model.nu)
terrain_ids = [model.geom(f"course_up_{i}").id for i in range(4)]
assert np.all(model.geom_contype[terrain_ids] == 1)
assert np.all(model.geom_conaffinity[terrain_ids] == 1)
print(f"Actor: {model.nu} actuators. Terrain adds {model.ngeom - actor_model.ngeom} static geoms.")

## 3. Define the goal

Every row is a **target**, excluding the spawn. The tracked body origin must enter each target's horizontal disk in order; the radius is inclusive. Progress advances by at most one target per control step. A target crossed between samples is not counted. This minimal evaluator does not check corridor width, falls, speed, jumping or foot support. Add task-specific criteria before using it as an obstacle-course benchmark.

The deadline uses Gymnasium `TimeLimit`: reaching the last target terminates; exceeding the deadline truncates.

In [ ]:
waypoints = np.array([
    [0, -1.1], [0, -2.3], [1.05, -2.3], [2.1, -2.3],
    [2.1, -3.1], [2.1, -3.8], [2.1, -4.9], [2.1, -5.5], [2.1, -6.4],
])
arrival_radius = .12
initial_qpos = model.qpos0.copy()
root_joint = np.flatnonzero(model.jnt_type == mujoco.mjtJoint.mjJNT_FREE)[0]
root_adr = model.jnt_qposadr[root_joint]
initial_qpos[root_adr:root_adr + 2] = [0, 0]

env = WaypointEnv(model, waypoints, body_name="pelvis",
                  arrival_radius=arrival_radius, initial_qpos=initial_qpos,
                  frame_skip=5, render_mode="rgb_array")
env = gym.wrappers.TimeLimit(env, max_episode_steps=round(90 / (model.opt.timestep * env.unwrapped.frame_skip)))
obs, info = env.reset(seed=42)
print("Observation:", obs.shape, "Controls:", env.action_space.shape)

## 4. Inspect the compiled scene and path

The figure is rendered from the newly compiled model. Disks show the goal tolerance; they do not prescribe feet or body poses.

In [ ]:
camera = mujoco.MjvCamera()
camera.lookat[:] = [1, -3, .4]
camera.distance, camera.azimuth, camera.elevation = 10, 140, -45
with mujoco.Renderer(model, height=480, width=640) as renderer:
    renderer.update_scene(env.unwrapped.data, camera=camera)
    frame = renderer.render()
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(frame)
axes[0].axis("off")
axes[0].set_title("Freshly compiled MyoFullBody scene")
route = np.vstack([[0, 0], waypoints])
axes[1].plot(route[:, 0], route[:, 1], "o-", color="teal")
for i, point in enumerate(waypoints):
    axes[1].add_patch(plt.Circle(point, arrival_radius, color="orange", alpha=.35))
    axes[1].annotate(str(i + 1), point)
axes[1].set(aspect="equal", xlabel="X (m)", ylabel="Y (m)", title="Ordered targets / 12 cm radius")
plt.show()

## 5. Step with a controller

A controller receives the observation and returns one raw control per actuator. Observations contain `qpos`, `qvel`, muscle activations, the next target's XY offset, remaining target count and arrival radius. Reward is minus distance to the target active at the start of the step. Read `info["solved"]` for completion.

This short zero-control smoke test checks physical stepping only; it does not solve the course. Replace the control line with your compatible controller. A TERRA adapter additionally needs its matching actor, observation normalization and a continuous full-body reference planner.

In [ ]:
for _ in range(10):
    action = np.zeros(env.action_space.shape, dtype=np.float32)
    obs, reward, terminated, truncated, info = env.step(action)
    assert np.isfinite(obs).all()
    if terminated or truncated:
        break
assert not info["solved"]
print({key: info[key] for key in ("solved", "next_waypoint", "distance")})
env.close()

## 6. Verify goal completion with a simple controlled actor

The same evaluator accepts a planar motor-driven actor. This small PD controller completes three ordered targets under MuJoCo dynamics. It is a functional example of the controller interface, not a muscle-locomotion solution.

In [ ]:
slider = mujoco.MjModel.from_xml_string("""
<mujoco><option timestep="0.01" gravity="0 0 0"/>
<worldbody><body name="actor" pos="0 0 .1">
<joint name="x" type="slide" axis="1 0 0" damping="1"/>
<joint name="y" type="slide" axis="0 1 0" damping="1"/>
<geom type="sphere" size=".05" mass="1"/>
</body></worldbody><actuator>
<motor joint="x" ctrlrange="-10 10"/><motor joint="y" ctrlrange="-10 10"/>
</actuator></mujoco>""")
probe = gym.wrappers.TimeLimit(
    WaypointEnv(slider, [[.3, 0], [.3, .3], [0, .3]], body_name="actor",
                arrival_radius=.025, frame_skip=1), max_episode_steps=1500)
probe.reset(seed=42)
positions = []
for _ in range(1500):
    delta = probe.unwrapped.get_obs_dict()["target_delta"]
    controls = 20 * delta - 7 * probe.unwrapped.data.qvel
    _, _, terminated, truncated, info = probe.step(controls)
    positions.append(probe.unwrapped.data.qpos.copy())
    if terminated or truncated:
        break
assert info["solved"]
print("All three targets completed at", round(probe.unwrapped.data.time, 2), "seconds")
positions = np.asarray(positions)
plt.figure(figsize=(5, 4))
plt.plot(positions[:, 0], positions[:, 1])
plt.axis("equal")
plt.title("Actual controlled planar trajectory")
plt.show()
probe.close()

## 7. Compose task variants

Vary the scene recipe, the actor spec and the goal separately. The example below compiles two scene variants, each with two arrival tolerances. No environment registration is required for every combination. Compilation validates MuJoCo structure, not reachability or policy success. If changing gap locations, update the waypoint coordinates too.

In [ ]:
from functools import partial
from itertools import product

for rise, radius in product((.04, .06), (.12, .2)):
    variant, _ = (ModelBuilder.from_spec(actor_spec)
                  .apply_transform(partial(course_scene, rise=rise)).build())
    trial = WaypointEnv(variant, waypoints, arrival_radius=radius, initial_qpos=initial_qpos)
    observation, _ = trial.reset(seed=42)
    assert trial.observation_space.contains(observation)
    trial.close()
    print(f"Compiled: step rise={rise:.2f} m, arrival radius={radius:.2f} m")

## Prior TERRA solution evidence

The recorded prototype completed this 8.5 m course in **37.93 s** with one TERRA checkpoint and a composed reference, without resetting the simulation. It used four 6 cm ascending steps, 36 cm beams and two 20 cm gaps. Both jumps passed independent takeoff/airborne/landing checks (0.12 s and 0.19 s airborne).

This is **one tuned course and one seed**. That earlier harness also checked a 35 cm path corridor and falls, and used the checkpoint's exact actor configuration. The notebook above does not reproduce that solver or those stronger criteria. No arbitrary-path capability or GPU parity is claimed. See [provenance and design choices](files/4.4/README.md).

![Verified course frames](files/4.4/course-controlled-frames.png)

In [ ]:
display(Video(filename=str(assets / "course-controlled.mp4"), embed=True, width=800))